# YemenVision-MTMC — المرحلة ٦: ماركة المركبة وطرازها وسنوات الإنتاج (مرسيدس، كورولا، دايو...)

يبني على دفتر المرحلة ٥ ونتائج تشغيله الفعلية.

## ما قالته نتائج المرحلة ٥ (ولماذا شكل هذا الدفتر كما هو)
| النتيجة | الدلالة |
|---|---|
| الكاشف اجتاز بوابته: Δ mAP50-95 = **+0.014** (CI +0.002…+0.026)، والمجمّع +0.058، والاسترجاع 0.425→0.500 | تحسّن حقيقي لكنه **صغير**. على الاختبار «النظيف» (252 صورة بلا شبيه في التدريب) الفاصل يشمل الصفر (−0.004…+0.055) |
| دُرّب الكاشف عند **640** رغم أن 49% من الأجسام <32px | رفع الدقة إلى 960 هو الرافع الأرجح التالي (انظر «إكمال المرحلة ٥» أدناه) |
| فئات `suv` (AP50 0.26) و`van` (0.28) أضعف الفئات | التباس بصري بين الطرازات — وهذا ما يعالجه هذا الدفتر بتصنيف دقيق على مستوى المركبة |
| فئات بلا اختبار كافٍ (garbagevan, scooter, policecar, minibus) | لا حكم عليها |
| تدريب ReID وصل epoch 20/30 ثم انقطع (train_acc≈0.999) | يُستأنف من Drive؛ لم يُقيَّم بعد |
| 11 ملف VeRi وهمي ما زال في Drive | أعد خلية الفحص بـ `QUARANTINE_FAKE_FILES=True` |
| VeRi الحقيقية فيها `test_label.xml` (11579 صورة) | قسم السمات (لون/نوع) صار ممكناً بنزاهة |

## ما يمكن وما لا يمكن
- **يمكن:** نوع المركبة (من الكاشف) + الماركة + الطراز + **مدى سنوات الجيل** (مثل 2005–2015) + اللون/النوع العام (من المرحلة ٥).
- **لا يمكن:** السنة الدقيقة، أو الفئة التجارية (trim)، أو المحرك. السنة تُقدَّر كجيل لأن مركبات السنوات المتجاورة تتشابه جداً.
- **المصنّف الجاهز (VMMRdb، EfficientNet-B4)** دقته المعلنة Top-1 ≈ 50% فقط على الماركة+الطراز+السنة، وبياناته أمريكية الطابع (1950–2016): قد يعرف مرسيدس وتويوتا، لكن حافلات **دايو** المحلية ومركبات السوق اليمني **على الأرجح خارج مفرداته**. لذلك الدفتر يقيس ما يعرفه فعلاً، ثم يتكيّف مع صورك.
- **الشرط الجوهري:** صور مقصوصة **مسمّاة بيدك** لكل فئة (≥30 يُفضَّل، ≥8 حداً أدنى). بدونها لا يمكن تعريف دايو ولا قياس شيء. الدفتر يقصّ لك المركبات من صورك بالكاشف لتسهيل التسمية (الخلية 3).

## الخطوات
1. الخلايا 1–2. 2. (اختياري) ضع مجلد صور كاملة في `images_to_crop_dir` ثم الخلية 3 لتحصل على مقصوصات مرتبة حسب نوع الكاشف.
3. صنّفها يدوياً في `vehicle_makemodel_crops/<الماركة>__<الطراز>__<سنةمن-سنةإلى>/` (مثال: `Toyota__Hilux__2005-2015`، `Daewoo__BS106`).
4. الخلايا 4–9: تحميل المصنّف، تقسيم بلا تسرب، قياس الجاهز، تكييف رأس خاص بك، معايرة «غير معروف»، مقارنة CLIP، ثم تقرير وترقية آمنة.

لا يُرقّى شيء إلا إن اجتاز البوابة (حد أدنى لدقة الطراز بفاصل ثقة، وعدم التراجع أمام الجاهز، ودقة عند الإجابة). المركبة غير الواثق منها تُعاد **«غير معروفة»** بدل تخمين اسم خاطئ.

In [ ]:
# ١. الإعداد (مستقل: يعمل في جلسة جديدة) — Drive + مساراته نفسها في دفاتر المراحل السابقة
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, math, time, random, shutil, hashlib, subprocess, importlib.util, warnings, copy, difflib
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

def status(kind, message):
    print({'ok': '✅', 'pause': '⏸️', 'error': '❌', 'warn': '⚠️'}.get(kind, 'ℹ️'), message)

try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC'); LOCAL = Path('/content/stage6_work')
except Exception:
    IN_COLAB = False
    DRIVE_ROOT = Path(os.environ.get('YV_DRIVE_ROOT', './YemenVision_MTMC')); LOCAL = Path(os.environ.get('YV_LOCAL', './stage6_work'))
    status('warn', 'لا يعمل داخل Colab؛ سيُستخدم مجلد محلي')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = DRIVE_ROOT / 'stage6' / RUN_ID; PROD_DIR = DRIVE_ROOT / 'production'
for d in (DRIVE_ROOT, LOCAL, RUN_DIR): d.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for c in iter(lambda: f.read(8 << 20), b''): h.update(c)
    return h.hexdigest()
def atomic_copy(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.tmp'); shutil.copyfile(src, tmp); os.replace(tmp, dst)
def _jd(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_jd), encoding='utf-8')

missing = [s for m, s in {'timm': 'timm', 'huggingface_hub': 'huggingface_hub', 'transformers': 'transformers', 'sklearn': 'scikit-learn',
                          'scipy': 'scipy', 'PIL': 'pillow', 'ultralytics': 'ultralytics>=8.3.0'}.items() if importlib.util.find_spec(m) is None]
if missing:
    print('تثبيت:', missing); subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
HAS_CUDA = torch.cuda.is_available(); DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')
status('ok' if HAS_CUDA else 'warn', f'torch={torch.__version__} | {"GPU: " + torch.cuda.get_device_name(0) if HAS_CUDA else "لا GPU — فعّل T4"}')
status('ok', f'Drive: {DRIVE_ROOT} | الجولة: {RUN_ID}')


In [ ]:
# ٢. الإعدادات — عدّلها قبل التشغيل
CONFIG6 = {
    'seed': 42,
    'hf_repo': 'Jordo23/vehicle-classifier',          # EfficientNet-B4 على VMMRdb (ماركة+طراز+سنة)؛ دقة Top-1 المعلنة ≈ 50% فقط
    'clip_model': 'openai/clip-vit-base-patch16',     # للتصنيف الصفري على قائمة فئاتك (يُجرَّب ولا يُعتمد تلقائياً)
    # صور مقصوصة مسمّاة بيدك: مجلد لكل فئة باسم  الماركة__الطراز__السنةمن-السنةإلى   (السنة اختيارية)
    # مثال: Toyota__Hilux__2005-2015 | Daewoo__BS106 | Mercedes-Benz__Actros__2010-2018
    'crops_dir': str(DRIVE_ROOT / 'vehicle_makemodel_crops'),
    # صور كاملة غير مسمّاة → نقصّ منها المركبات بالكاشف لتسهّل عليك التسمية (اتركه فارغاً إن لا حاجة)
    'images_to_crop_dir': '',
    'min_crop_side_px': 64,
    'min_images_per_class': 8,       # أقل من ذلك: الفئة «غير موثوقة» ولا تدخل الاختبار
    'val_frac': 0.15, 'test_frac': 0.25, 'neardup_hamming': 6,
    'unknown_target_precision': 0.90,  # عتبة «غير معروف»: نرفض التنبؤ حين لا نبلغ هذه الدقة على التحقق
    'epochs_head': 80, 'boot': 1000,
    'multiword_makes': ['land rover', 'alfa romeo', 'aston martin', 'mercedes benz', 'rolls royce', 'great wall', 'range rover'],
    # أسماء شائعة عندك ← الاسم المعياري في VMMRdb (أضف ما يلزم)
    'aliases': {'mercedes': 'mercedes benz', 'benz': 'mercedes benz', 'mercedesbenz': 'mercedes benz', 'landcruiser': 'land cruiser',
                'cresida': 'cressida', 'corola': 'corolla'},
}
random.seed(CONFIG6['seed']); np.random.seed(CONFIG6['seed']); torch.manual_seed(CONFIG6['seed'])
save_json(RUN_DIR / 'config.json', CONFIG6)
CROPS_DIR = Path(CONFIG6['crops_dir']); CROPS_DIR.mkdir(parents=True, exist_ok=True)
status('ok', f'مجلد الفئات المسمّاة: {CROPS_DIR}')


In [ ]:
# ٣. مساعد التسمية: يقصّ المركبات من صورك الكاملة بالكاشف (production) ويرتّبها حسب نوع الكاشف ليسهل عليك وضعها في مجلدات الماركة/الطراز
# التسمية تتم بيدك (أو بخبير بالمركبات المحلية). الدفتر لا يخمّن تسميات ولا يدرّب على مخرجات نموذج غير مراجعة.
TO_LABEL = RUN_DIR / 'to_label'
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
if CONFIG6['images_to_crop_dir'] and Path(CONFIG6['images_to_crop_dir']).is_dir():
    det_path = PROD_DIR / 'detector_best.pt'
    if not det_path.is_file():                      # كاشف المرحلة ٥ الذي اجتاز بوابته ولم يُرقَّ بعد
        for rep in sorted((DRIVE_ROOT / 'stage5').glob('*/detector_final_eval.json'), reverse=True):
            try:
                r = json.loads(rep.read_text('utf-8'))
                if r.get('pass') and Path(r['path']).is_file(): det_path = Path(r['path']); break
            except Exception: pass
    if not det_path.is_file(): det_path = next(iter([p for p in (DRIVE_ROOT / 'detection_yolo11s_best.pt', DRIVE_ROOT / 'best.pt') if p.is_file()]), None)
    if det_path is None:
        status('pause', 'لا كاشف في Drive لاستخدامه في القص')
    else:
        from ultralytics import YOLO
        det = YOLO(str(det_path)); n = 0
        paths = [p for p in sorted(Path(CONFIG6['images_to_crop_dir']).rglob('*')) if p.suffix.lower() in IMG_EXT]
        for i in range(0, len(paths), 32):
            for res in det.predict(source=[str(p) for p in paths[i:i + 32]], conf=0.4, verbose=False, stream=True, half=HAS_CUDA, device=0 if HAS_CUDA else 'cpu'):
                im = Image.open(res.path).convert('RGB')
                for j, (box, c) in enumerate(zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy())):
                    x1, y1, x2, y2 = [int(v) for v in box]
                    if min(x2 - x1, y2 - y1) < CONFIG6['min_crop_side_px']: continue
                    out = TO_LABEL / str(det.names[int(c)]) / f'{Path(res.path).stem}_{j}.jpg'; out.parent.mkdir(parents=True, exist_ok=True)
                    im.crop((x1, y1, x2, y2)).save(out, quality=95); n += 1
        status('ok', f'{n} مقصوصة في {TO_LABEL} — انقلها إلى {CROPS_DIR}/<الماركة>__<الطراز>__<السنوات>/ بعد التسمية')
else:
    status('pause', 'لا مجلد صور للقص (اختياري)')


In [ ]:
# ٤. المصنّف الجاهز على VMMRdb + بناء هرم (ماركة ← طراز ← سنة) — احتمالات الماركة/الطراز تُجمع من كل السنوات
from scipy import sparse

def norm_text(s):
    s = str(s).lower().replace('_', ' ').replace('-', ' ')
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', s)).strip()

def parse_vmmr_name(name):
    toks = norm_text(name); year = None
    m = re.search(r'\b((?:19|20)\d{2})\s*$', toks)
    if m: year = int(m.group(1)); toks = toks[:m.start()].strip()
    make = next((mw for mw in CONFIG6['multiword_makes'] if toks.startswith(mw + ' ') or toks == mw), None)
    if make is None: make = toks.split(' ')[0] if toks else ''
    return make, toks[len(make):].strip(), year

def build_hierarchy(class_names):
    parsed = [parse_vmmr_name(n) for n in class_names]
    makes = sorted({p[0] for p in parsed}); mms = sorted({(p[0], p[1]) for p in parsed})
    mk_i = {m: i for i, m in enumerate(makes)}; mm_i = {m: i for i, m in enumerate(mms)}
    C = len(parsed); rows = np.arange(C)
    mk_of = np.array([mk_i[p[0]] for p in parsed]); mm_of = np.array([mm_i[(p[0], p[1])] for p in parsed])
    return {'makes': makes, 'mms': mms, 'mk_of': mk_of, 'mm_of': mm_of, 'year_of': np.array([p[2] if p[2] else 0 for p in parsed]),
            'M_make': sparse.csr_matrix((np.ones(C, np.float32), (rows, mk_of)), shape=(C, len(makes))),
            'M_mm': sparse.csr_matrix((np.ones(C, np.float32), (rows, mm_of)), shape=(C, len(mms))), 'mk_i': mk_i, 'mm_i': mm_i}

def marginalize(probs, H):
    # probs (N,C) → احتمال الماركة (N,Mk) والطراز (N,Mm): جمع كل السنوات. خطأ السنة لا يُفسد الماركة/الطراز
    return np.asarray(H['M_make'].T @ probs.T).T, np.asarray(H['M_mm'].T @ probs.T).T

def year_estimate(prob_row, mm_id, H, mass=0.8):
    idx = np.where((H['mm_of'] == mm_id) & (H['year_of'] > 0))[0]
    if len(idx) == 0: return None
    p = prob_row[idx] / max(prob_row[idx].sum(), 1e-12); order = np.argsort(H['year_of'][idx]); yrs = H['year_of'][idx][order]; c = np.cumsum(p[order])
    lo = yrs[np.searchsorted(c, (1 - mass) / 2)]; hi = yrs[min(np.searchsorted(c, 1 - (1 - mass) / 2), len(yrs) - 1)]
    return {'median': int(yrs[min(np.searchsorted(c, 0.5), len(yrs) - 1)]), 'range': [int(lo), int(hi)], 'mass': mass}

def parse_user_label(folder_name):
    parts = folder_name.split('__'); make = norm_text(parts[0]); model = norm_text(parts[1]) if len(parts) > 1 else ''
    make = CONFIG6['aliases'].get(make.replace(' ', ''), CONFIG6['aliases'].get(make, make)); model = CONFIG6['aliases'].get(model.replace(' ', ''), model)
    y0 = y1 = None
    if len(parts) > 2:
        ys = re.findall(r'(?:19|20)\d{2}', parts[2]); y0 = int(ys[0]) if ys else None; y1 = int(ys[-1]) if ys else y0
    return {'folder': folder_name, 'make': make, 'model': model, 'y0': y0, 'y1': y1}

def match_to_vocab(lab, H):
    # يربط (ماركة، طراز) المستخدم بمفردات VMMRdb. غير الموجود = خارج المفردات (OOV) ويُعلَن صراحةً — مثل حافلات دايو المحلية
    mk = lab['make']
    if mk not in H['mk_i']:
        c = difflib.get_close_matches(mk, H['makes'], n=1, cutoff=0.85); mk = c[0] if c else None
    if mk is None: return None, None
    models = [m for (a, m) in H['mms'] if a == mk]
    key = lab['model']; hit = None
    if key in models: hit = key
    else:
        flat = {m.replace(' ', ''): m for m in models}
        if key.replace(' ', '') in flat: hit = flat[key.replace(' ', '')]
        else:
            c = difflib.get_close_matches(key, models, n=1, cutoff=0.8); hit = c[0] if c else None
    return H['mk_i'][mk], (H['mm_i'][(mk, hit)] if hit is not None else None)

def load_vmmr_classifier():
    import timm
    from huggingface_hub import hf_hub_download
    model = timm.create_model('hf-hub:' + CONFIG6['hf_repo'], pretrained=True).to(DEVICE).eval()
    cfg = timm.data.resolve_data_config({}, model=model); tf = timm.data.create_transform(**cfg)
    path = hf_hub_download(CONFIG6['hf_repo'], 'class_mapping.csv'); df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    if {'make', 'model', 'year'} <= set(cols): names = (df[cols['make']].astype(str) + ' ' + df[cols['model']].astype(str) + ' ' + df[cols['year']].astype(str)).tolist(); ids = df.iloc[:, 0].astype(int).values
    else:
        idc = next((cols[k] for k in ('class_id', 'id', 'idx', 'index') if k in cols), df.columns[0])
        nmc = next((cols[k] for k in ('class_name', 'name', 'label', 'class') if k in cols), [c for c in df.columns if df[c].dtype == object][0])
        ids, names = df[idc].astype(int).values, df[nmc].astype(str).tolist()
    ordered = [n for _, n in sorted(zip(ids, names))]
    n_out = model.get_classifier().out_features
    if len(ordered) != n_out: raise RuntimeError(f'عدد الفئات في class_mapping.csv ({len(ordered)}) لا يطابق مخرجات النموذج ({n_out})')
    return model, tf, cfg, ordered

VMMR = {}
if HAS_CUDA:
    try:
        vmodel, vtf, vcfg, vnames = load_vmmr_classifier(); VH = build_hierarchy(vnames)
        VMMR = {'model': vmodel, 'tf': vtf, 'names': vnames, 'H': VH}
        print('إعداد المعالجة المسبقة للنموذج:', {k: vcfg[k] for k in ('input_size', 'mean', 'std') if k in vcfg})
        status('ok', f'{len(vnames)} فئة | {len(VH["makes"])} ماركة | {len(VH["mms"])} طراز. أمثلة: {vnames[:3]}')
        print('تنبيه: بيانات VMMRdb أمريكية الطابع (1950–2016). الدقة المعلنة Top-1≈50% على الماركة+الطراز+السنة؛ نقيسها على صورك أنت أدناه.')
    except Exception as exc:
        status('error', f'تعذر تحميل المصنّف: {type(exc).__name__}: {exc}')
else:
    status('pause', 'يلزم GPU')


In [ ]:
# ٥. مجموعتك المسمّاة: تقسيم بلا تسرب (مجموعات شبه المكرر داخل كل فئة) — الاختبار لا يُستخدم في أي قرار
POPCOUNT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def dhash(path, size=8):
    with Image.open(path) as im:
        g = im.convert('L').resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16); return np.packbits((a[:, 1:] > a[:, :-1]).flatten())
def hamming_matrix(A, B): return POPCOUNT[np.bitwise_xor(A[:, None, :], B[None, :, :])].sum(-1)
def union_find_groups(n, pairs):
    parent = list(range(n))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i, j in pairs:
        a, b = find(i), find(j)
        if a != b: parent[b] = a
    g = {}
    for i in range(n): g.setdefault(find(i), []).append(i)
    return list(g.values())

def split_class_groups(n_images, groups, test_frac, val_frac, seed):
    # يعيد 'train'/'val'/'test' لكل صورة داخل فئة واحدة، بمجموعات كاملة
    rng = np.random.default_rng(seed); order = rng.permutation(len(groups)); lab = ['train'] * n_images
    if len(groups) < 3: return lab, False
    need_te, need_va = max(1, round(test_frac * n_images)), max(1, round(val_frac * n_images)); te = va = 0
    cap_te, cap_va = max(need_te * 1.5, need_te + 1), max(need_va * 1.5, need_va + 1)   # مجموعة ضخمة لا تبتلع الفئة كلها
    for gi in order:
        n_train_left = sum(1 for l in lab if l == 'train'); g = groups[gi]
        if te < need_te and te + len(g) <= cap_te and n_train_left - len(g) >= n_images * 0.5:
            for i in g: lab[i] = 'test'
            te += len(g)
        elif va < need_va and va + len(g) <= cap_va and n_train_left - len(g) >= n_images * 0.5:
            for i in g: lab[i] = 'val'
            va += len(g)
    return lab, ('test' in lab and 'train' in lab)

DATA = None
rows = []
for d in sorted(p for p in CROPS_DIR.iterdir() if p.is_dir()) if CROPS_DIR.is_dir() else []:
    lab = parse_user_label(d.name)
    for p in sorted(d.rglob('*')):
        if p.suffix.lower() in IMG_EXT: rows.append({'path': str(p), **lab})
if not rows:
    status('pause', f'لا صور مسمّاة في {CROPS_DIR}. هذا هو النقص الجوهري: بدونها لا يمكن تعريف دايو/هايلكس/كورولا... ولا قياس الدقة. أنشئ مجلداً لكل فئة (≥{CONFIG6["min_images_per_class"]} صور متنوعة، يُفضّل ≥30).')
else:
    DATA = pd.DataFrame(rows); DATA['mmkey'] = DATA.make + ' ' + DATA.model
    t0 = time.time(); DATA['hash'] = [dhash(p) for p in DATA.path]; split = np.array(['train'] * len(DATA), dtype=object); reliable = {}
    for folder, idx in DATA.groupby('folder').indices.items():
        H8 = np.stack(DATA.hash.values[idx]); d = hamming_matrix(H8, H8)
        pairs = [(a, b) for a, b in np.argwhere(np.triu(d <= CONFIG6['neardup_hamming'], 1))]
        groups = union_find_groups(len(idx), pairs)
        lab_, ok = split_class_groups(len(idx), groups, CONFIG6['test_frac'], CONFIG6['val_frac'], CONFIG6['seed'])
        split[idx] = lab_; reliable[folder] = bool(ok and len(idx) >= CONFIG6['min_images_per_class'])
    DATA['split'] = split; DATA['reliable'] = DATA.folder.map(reliable)
    summ = DATA.groupby('folder').agg(n=('path', 'count'), train=('split', lambda s: (s == 'train').sum()), val=('split', lambda s: (s == 'val').sum()),
                                      test=('split', lambda s: (s == 'test').sum()), reliable=('reliable', 'first'))
    print(summ.to_string()); weak = summ.index[~summ.reliable].tolist()
    status('warn' if weak else 'ok', f'فئات غير موثوقة (صور/مجموعات قليلة، لا تدخل الاختبار): {weak}')
    status('ok', f'{len(DATA)} صورة | {DATA.folder.nunique()} فئة | بصمات في {time.time()-t0:.0f}s')
    DATA.drop(columns=['hash']).to_csv(RUN_DIR / 'dataset_split.csv', index=False)


In [ ]:
# ٦. استخراج الميزات والاحتمالات من المصنّف الجاهز + قياسه على صورك (الماركة / الطراز / السنة)
from torch.utils.data import Dataset, DataLoader
class _PIL(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = list(paths), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im: return self.tf(im.convert('RGB'))

@torch.no_grad()
def extract_vmmr(paths, batch=32):
    m = VMMR['model']; feats, probs = [], []
    for x in DataLoader(_PIL(paths, VMMR['tf']), batch_size=batch, num_workers=2):
        x = x.to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16, enabled=HAS_CUDA):
            f = m.forward_head(m.forward_features(x), pre_logits=True); lg = m.get_classifier()(f)
        feats.append(f.float().cpu()); probs.append(F.softmax(lg.float(), 1).cpu())
    return torch.cat(feats).numpy(), torch.cat(probs).numpy()

def boot_ci(x, n=1000, seed=0):
    x = np.asarray(x, float); rng = np.random.default_rng(seed); m = x[rng.integers(0, len(x), (n, len(x)))].mean(1)
    return float(x.mean()), float(np.quantile(m, 0.025)), float(np.quantile(m, 0.975))

FEAT, PRE = {}, {}
if DATA is not None and VMMR:
    FEATS, PROBS = extract_vmmr(DATA.path.values); H = VMMR['H']
    # ربط فئاتك بمفردات VMMRdb
    vm = {f: match_to_vocab(parse_user_label(f), H) for f in DATA.folder.unique()}
    DATA['vm_make'] = DATA.folder.map(lambda f: vm[f][0] if vm[f][0] is not None else -1); DATA['vm_mm'] = DATA.folder.map(lambda f: vm[f][1] if vm[f][1] is not None else -1)
    cov = pd.DataFrame({'folder': list(vm), 'make_in_vocab': [v[0] is not None for v in vm.values()], 'model_in_vocab': [v[1] is not None for v in vm.values()]})
    print('تغطية مفردات VMMRdb لفئاتك:'); print(cov.to_string(index=False))
    status('warn' if (~cov.model_in_vocab).any() else 'ok', f'فئات خارج المفردات (المصنّف الجاهز لا يعرفها أصلاً): {cov.folder[~cov.model_in_vocab].tolist()}')
    pm, pmm = marginalize(PROBS, H)
    te = (DATA.split == 'test') & DATA.reliable; inv = te & (DATA.vm_mm >= 0)
    def acc(mask, pred, truth): return float((pred[mask.values] == truth[mask.values]).mean()) if mask.sum() else float('nan')
    pre_mm_ok = (pmm.argmax(1) == DATA.vm_mm.values)
    top5 = np.argsort(-pmm, 1)[:, :5]; pre_mm_top5 = np.array([DATA.vm_mm.values[i] in top5[i] for i in range(len(DATA))])
    res = {'n_test': int(te.sum()), 'n_test_in_vocab': int(inv.sum()),
           'make_acc_in_vocab': acc(inv, pm.argmax(1), DATA.vm_make.values), 'model_acc_in_vocab': acc(inv, pmm.argmax(1), DATA.vm_mm.values),
           'model_top5_in_vocab': float(pre_mm_top5[inv.values].mean()) if inv.sum() else float('nan'),
           'model_acc_all_test_(OOV_counts_wrong)': float(pre_mm_ok[te.values].mean()) if te.sum() else float('nan')}
    yr = []
    for i in np.where(inv.values)[0]:
        y = year_estimate(PROBS[i], int(pmm[i].argmax()), H)
        if y and DATA.y0.iloc[i] == DATA.y0.iloc[i] and DATA.y0.iloc[i] is not None and not pd.isna(DATA.y0.iloc[i]): yr.append(DATA.y0.iloc[i] - 2 <= y['median'] <= DATA.y1.iloc[i] + 2)
    res['year_within_label_range_±2'] = float(np.mean(yr)) if yr else None
    PRE = {'res': res, 'pre_mm_ok': pre_mm_ok, 'inv': inv.values, 'pm': pm, 'pmm': pmm}
    print(json.dumps(res, ensure_ascii=False, indent=1)); save_json(RUN_DIR / 'pretrained_eval.json', res)
    if inv.sum() < 30: status('warn', f'{int(inv.sum())} صورة اختبار داخل المفردات فقط — الأرقام أعلاه استرشادية وليست حكماً')
else:
    status('pause', 'القياس يحتاج صوراً مسمّاة ومصنّفاً محمّلاً')


In [ ]:
# ٧. التكيّف مع مركباتك: رأس تصنيف على ميزات EfficientNet-B4 المجمَّدة + معايرة + رفض «غير معروف»
GATES6 = {'min_test_top1_ci_low': 0.50,        # الحد الأدنى لفاصل الثقة السفلي لدقة الطراز على اختبارك (عدّله بحسب حاجتك)
          'max_inv_regression': 0.02,          # أقصى تراجع مسموح أمام المصنّف الجاهز على الفئات التي يعرفها
          'unknown_precision_slack': 0.05}
HEAD, HEAD_RESULT = {}, {}
if DATA is not None and VMMR and 'FEATS' in globals():
    classes = sorted(DATA.folder.unique()); cid = {c: i for i, c in enumerate(classes)}; K = len(classes)
    y = DATA.folder.map(cid).values; Xn = FEATS / (np.linalg.norm(FEATS, axis=1, keepdims=True) + 1e-12)
    tr, va = (DATA.split == 'train').values, (DATA.split == 'val').values; te = ((DATA.split == 'test') & DATA.reliable).values
    cls_mm = np.array([DATA.mmkey[DATA.folder == c].iloc[0] for c in classes]); cls_mk = np.array([DATA.make[DATA.folder == c].iloc[0] for c in classes])
    if te.sum() < 20 or K < 2:
        status('pause', f'اختبار صغير جداً ({int(te.sum())}) أو فئة واحدة — أضف صوراً قبل الحكم')
    else:
        Xt, yt = torch.tensor(Xn[tr], device=DEVICE), torch.tensor(y[tr], device=DEVICE)
        cnt = np.bincount(y[tr], minlength=K).astype(np.float32); w = torch.tensor((1.0 / np.sqrt(np.maximum(cnt, 1))) / (1.0 / np.sqrt(np.maximum(cnt, 1))).mean(), device=DEVICE)
        head = nn.Linear(Xn.shape[1], K).to(DEVICE); opt = torch.optim.AdamW(head.parameters(), lr=2e-3, weight_decay=1e-2)
        for ep in range(CONFIG6['epochs_head']):
            perm = torch.randperm(len(Xt), device=DEVICE)
            for s in range(0, len(Xt), 256):
                idx = perm[s:s + 256]; loss = F.cross_entropy(head(Xt[idx] * 10), yt[idx], weight=w, label_smoothing=0.1)   # ×10: مقياس لوغاريتمات على ميزات مُطبَّعة
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): logits_all = (head(torch.tensor(Xn, device=DEVICE) * 10)).float()
        # معايرة درجة الحرارة على التحقق (إن وُجد)
        T = 1.0
        if va.sum() >= 20:
            lv, yv = logits_all[torch.tensor(va, device=DEVICE)], torch.tensor(y[va], device=DEVICE); logT = torch.zeros(1, device=DEVICE, requires_grad=True)
            o2 = torch.optim.LBFGS([logT], lr=0.1, max_iter=50)
            def _c(): o2.zero_grad(); l = F.cross_entropy(lv / logT.exp(), yv); l.backward(); return l
            o2.step(_c); T = float(logT.exp())
        P_all = F.softmax(logits_all / T, 1).cpu().numpy(); pred = P_all.argmax(1); conf = P_all.max(1)
        # عتبة «غير معروف»: أصغر عتبة تحقق دقة القبول المطلوبة على التحقق
        thr, thr_ok = 1.0, False
        if va.sum() >= 20:
            order = np.argsort(-conf[va]); ok = (pred[va] == y[va])[order]; prec = np.cumsum(ok) / (np.arange(len(ok)) + 1)
            good = np.where(prec >= CONFIG6['unknown_target_precision'])[0]
            if len(good): thr, thr_ok = float(conf[va][order][good.max()]), True
        # ---- الاختبار (يُلمس مرة واحدة) ----
        full_ok = (pred == y); mm_ok = (cls_mm[pred] == cls_mm[y]); mk_ok = (cls_mk[pred] == cls_mk[y])
        top5 = np.argsort(-P_all, 1)[:, :5]; top5_ok = np.array([y[i] in top5[i] for i in range(len(y))])
        m, lo, hi = boot_ci(mm_ok[te], CONFIG6['boot'])
        acc_mask = conf >= thr; sel = te & acc_mask
        HEAD_RESULT = {'n_test': int(te.sum()), 'n_classes': K, 'temperature': T, 'unknown_threshold': thr, 'threshold_reached_target_on_val': thr_ok,
                       'make_acc': float(mk_ok[te].mean()), 'model_acc': m, 'model_acc_ci': [lo, hi], 'full_label_acc(generation)': float(full_ok[te].mean()),
                       'top5_full': float(top5_ok[te].mean()), 'coverage_at_threshold': float(sel.sum() / te.sum()),
                       'precision_when_answering': float(mm_ok[sel].mean()) if sel.sum() else None}
        # مقارنة بالمصنّف الجاهز على ما يعرفه فقط
        inv_te = PRE['inv'] & te
        if inv_te.sum() >= 30:
            d = mm_ok[inv_te].astype(float) - PRE['pre_mm_ok'][inv_te].astype(float); dm, dlo, dhi = boot_ci(d, CONFIG6['boot'])
            HEAD_RESULT['vs_pretrained_in_vocab'] = {'n': int(inv_te.sum()), 'head_acc': float(mm_ok[inv_te].mean()), 'pretrained_acc': float(PRE['pre_mm_ok'][inv_te].mean()), 'delta': dm, 'ci': [dlo, dhi]}
        oov_te = te & (DATA.vm_mm.values < 0)
        if oov_te.sum(): HEAD_RESULT['oov_classes_head_acc'] = {'n': int(oov_te.sum()), 'head_acc': float(mm_ok[oov_te].mean()), 'pretrained_acc': 0.0}
        per = pd.DataFrame({'folder': np.array(classes)[y[te]], 'ok': mm_ok[te]}).groupby('folder').ok.agg(['mean', 'count']).round(3)
        print('دقة الطراز لكل فئة على الاختبار:'); print(per.to_string()); print(json.dumps(HEAD_RESULT, ensure_ascii=False, indent=1, default=_jd))
        reasons = []
        if not lo >= GATES6['min_test_top1_ci_low']: reasons.append(f'الحد السفلي لدقة الطراز {lo:.3f} < {GATES6["min_test_top1_ci_low"]}')
        v = HEAD_RESULT.get('vs_pretrained_in_vocab')
        if v and v['delta'] < -GATES6['max_inv_regression']: reasons.append('أسوأ من المصنّف الجاهز على فئاته')
        if not thr_ok: reasons.append('لا عتبة «غير معروف» تبلغ الدقة المطلوبة على التحقق (التحقق صغير أو الفئات متقاربة)')
        elif HEAD_RESULT['precision_when_answering'] is not None and HEAD_RESULT['precision_when_answering'] < CONFIG6['unknown_target_precision'] - GATES6['unknown_precision_slack']:
            reasons.append('دقة الإجابة على الاختبار أقل من المستهدف')
        HEAD_RESULT.update({'pass': not reasons, 'fail_reasons': reasons}); HEAD = {'state': {k: v_.cpu() for k, v_ in head.state_dict().items()}, 'classes': classes, 'T': T, 'thr': thr}
        status('ok' if not reasons else 'warn', 'بوابة الماركة/الطراز: ' + ('PASS' if not reasons else 'FAIL — ' + ' | '.join(reasons)))
        save_json(RUN_DIR / 'makemodel_head_eval.json', HEAD_RESULT)
else:
    status('pause', 'التكيّف NOT_EVALUATED (يلزم صور مسمّاة + المصنّف + خلية القياس السابقة)')


In [ ]:
# ٨. مقارنة: التصنيف الصفري بـCLIP على قائمة فئاتك (بلا تدريب) — للفئات التي لا صور كافية لها بعد
CLIP_RESULT = {}
if HEAD and DATA is not None:
    from transformers import CLIPModel, CLIPProcessor
    cm = CLIPModel.from_pretrained(CONFIG6['clip_model']).to(DEVICE).eval().half(); cp = CLIPProcessor.from_pretrained(CONFIG6['clip_model'])
    mm_list = sorted(DATA.mmkey.unique()); texts = [f'a photo of a {m}, a vehicle' for m in mm_list]
    with torch.no_grad():
        tf_ = cm.get_text_features(**{k: v.to(DEVICE) for k, v in cp(text=texts, return_tensors='pt', padding=True).items()}); tf_ = F.normalize(tf_.float(), dim=-1)
        idx_te = np.where(te)[0]; preds = []
        for s in range(0, len(idx_te), 64):
            ims = [Image.open(DATA.path.iloc[i]).convert('RGB') for i in idx_te[s:s + 64]]
            f = cm.get_image_features(pixel_values=cp(images=ims, return_tensors='pt')['pixel_values'].to(DEVICE).half()); preds.append((F.normalize(f.float(), dim=-1) @ tf_.T).argmax(1).cpu().numpy())
    pred_mm = np.array(mm_list)[np.concatenate(preds)]; ok = (pred_mm == DATA.mmkey.values[idx_te]); m, lo, hi = boot_ci(ok, CONFIG6['boot'])
    CLIP_RESULT = {'model_acc': m, 'ci': [lo, hi], 'n': int(len(ok)), 'note': 'صفري: لا يرى صورك. للمقارنة فقط؛ لا يدخل الإنتاج تلقائياً'}
    print(f"CLIP صفري (الطراز): {m:.3f} (CI {lo:.3f}–{hi:.3f}) مقابل الرأس المتكيّف: {HEAD_RESULT['model_acc']:.3f}")
    save_json(RUN_DIR / 'clip_zeroshot_eval.json', CLIP_RESULT); del cm; torch.cuda.empty_cache()
else:
    status('pause', 'مقارنة CLIP NOT_EVALUATED')


In [ ]:
# ٩. وحدة الاستخدام (كاشف ← قص ← ماركة/طراز/سنوات) + تجميع إطارات المسار الواحد + التقرير والترقية الآمنة
PIPE_SRC = r'''
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image

def aggregate_track_probs(prob_list, weights=None):
    # تجميع إطارات نفس المركبة: متوسط لوغاريتم الاحتمالات (أمتن من تصويت الأغلبية مع إطارات ضعيفة)، وزن أكبر للصور الأكبر/الأوضح
    P = np.clip(np.stack([np.asarray(p, np.float64) for p in prob_list]), 1e-9, 1.0)
    w = np.ones(len(P)) if weights is None else np.asarray(weights, np.float64)
    lp = (np.log(P) * (w / w.sum())[:, None]).sum(0); e = np.exp(lp - lp.max())
    return e / e.sum()

def parse_folder(name):
    parts = name.split('__'); import re
    ys = re.findall(r'(?:19|20)\d{2}', parts[2]) if len(parts) > 2 else []
    return {'make': parts[0].replace('_', ' '), 'model': parts[1].replace('_', ' ') if len(parts) > 1 else '', 'years': [int(ys[0]), int(ys[-1])] if ys else None}

class VehicleDescriber:
    def __init__(self, bundle_path, device='cuda'):
        import timm
        b = torch.load(bundle_path, map_location='cpu', weights_only=False); self.b, self.dev = b, device
        self.backbone = timm.create_model('hf-hub:' + b['feature_model'], pretrained=True).to(device).eval()
        self.tf = timm.data.create_transform(**timm.data.resolve_data_config({}, model=self.backbone))
        self.head = nn.Linear(b['head']['weight'].shape[1], len(b['classes'])).to(device); self.head.load_state_dict(b['head']); self.head.eval()
    @torch.no_grad()
    def probs(self, pil_images):
        x = torch.stack([self.tf(im.convert('RGB')) for im in pil_images]).to(self.dev)
        f = self.backbone.forward_head(self.backbone.forward_features(x), pre_logits=True).float()
        f = F.normalize(f, dim=1)
        return F.softmax(self.head(f * 10) / self.b['T'], 1).cpu().numpy()
    def describe(self, prob, vehicle_type=None):
        order = np.argsort(-prob)[:3]; top = self.b['classes'][int(order[0])]; info = parse_folder(top); known = bool(prob[order[0]] >= self.b['thr'])
        return {'vehicle_type': vehicle_type, 'status': 'known' if known else 'unknown', 'make': info['make'] if known else None,
                'model': info['model'] if known else None, 'years': info['years'] if known else None, 'confidence': float(prob[order[0]]),
                'top3': [{'label': self.b['classes'][int(i)], 'p': float(prob[i])} for i in order]}
'''
exec(PIPE_SRC, globals())
(DRIVE_ROOT / 'yemenvision_vehicle_describer.py').write_text(PIPE_SRC, encoding='utf-8')

# تحقق رقمي صغير من تجميع المسار (بيانات اصطناعية): إطاران ضعيفان لا يقلبان إطاراً قوياً
_a = aggregate_track_probs([np.array([0.7, 0.2, 0.1]), np.array([0.4, 0.45, 0.15]), np.array([0.8, 0.1, 0.1])])
assert _a.argmax() == 0 and abs(_a.sum() - 1) < 1e-9

rows = []
def gate_row(name, res, ok, detail): rows.append({'الفحص': name, 'الحالة': 'PASS' if ok else ('NOT_EVALUATED' if not res else 'FAIL'), 'التفصيل': detail})
r = HEAD_RESULT
gate_row('ماركة/طراز على صورك', r, bool(r and r.get('pass')),
         (f"طراز {r['model_acc']:.3f} (CI {r['model_acc_ci'][0]:.3f}–{r['model_acc_ci'][1]:.3f}) | ماركة {r['make_acc']:.3f} | تغطية {r['coverage_at_threshold']:.2f} بدقة {r['precision_when_answering']}"
          + ('' if r['pass'] else ' | ' + ' | '.join(r['fail_reasons']))) if r else 'يلزم صور مسمّاة')
gate_row('المصنّف الجاهز (VMMRdb) على صورك', PRE.get('res'), bool(PRE), (f"طراز داخل المفردات {PRE['res']['model_acc_in_vocab']:.3f} (n={PRE['res']['n_test_in_vocab']})") if PRE else 'لم يُقيَّم')
gate_row('CLIP صفري (للمقارنة)', CLIP_RESULT, bool(CLIP_RESULT), f"{CLIP_RESULT['model_acc']:.3f}" if CLIP_RESULT else 'لم يُقيَّم')
report = pd.DataFrame(rows); print(report.to_string(index=False))

PROMOTED6 = {}
if HEAD and HEAD_RESULT.get('pass'):
    bundle = RUN_DIR / 'vehicle_makemodel.pt'
    torch.save({'head': HEAD['state'], 'classes': HEAD['classes'], 'T': HEAD['T'], 'thr': HEAD['thr'], 'feature_model': CONFIG6['hf_repo'],
                'run_id': RUN_ID, 'eval': HEAD_RESULT}, bundle)
    PROD_DIR.mkdir(parents=True, exist_ok=True); dst = PROD_DIR / 'vehicle_makemodel.pt'
    if dst.is_file(): atomic_copy(dst, DRIVE_ROOT / 'production_archive' / RUN_ID / dst.name)
    atomic_copy(bundle, dst); atomic_copy(DRIVE_ROOT / 'yemenvision_vehicle_describer.py', PROD_DIR / 'yemenvision_vehicle_describer.py')
    PROMOTED6 = {'vehicle_makemodel.pt': sha256_file(dst)}; status('ok', 'رُقّي vehicle_makemodel.pt (القديم، إن وُجد، في production_archive)')
else:
    status('warn', 'لم يُرقَّ شيء: لم يجتز الرأس بوابته أو لا بيانات. لا شيء في production/ تغيّر.')
save_json(RUN_DIR / 'stage6_report.json', {'run_id': RUN_ID, 'head': HEAD_RESULT, 'pretrained': PRE.get('res'), 'clip': CLIP_RESULT, 'promoted': PROMOTED6})
(RUN_DIR / 'stage6_report.md').write_text('# تقرير المرحلة ٦ — ' + RUN_ID + '\n\n' + report.to_string(index=False), encoding='utf-8')
print('\nمثال استخدام لاحقاً:\n  from yemenvision_vehicle_describer import VehicleDescriber, aggregate_track_probs\n  vd = VehicleDescriber("production/vehicle_makemodel.pt"); p = vd.probs([crop1, crop2]); print(vd.describe(aggregate_track_probs(list(p)), "pickup"))')


## إكمال المرحلة ٥ (قبل أو بعد هذا الدفتر)
1. **ReID:** في الخلية 1 من دفتر المرحلة ٥ ضع `RESUME_RUN_ID = '20261118T125424Z'` وأعد الخلايا حتى ReID؛ يستأنف من epoch 20 ويقيس ويُرقّي إن اجتاز.
2. **الكاشف:** مرشح ثانٍ بـ `'det_imgsz': 960` (و`det_batch` 4–6) لأن نصف الأجسام صغيرة. ثم قارنه بنفس الاختبار.
3. أعد تعريف مجموعات الأجسام النادرة (garbagevan, scooter, policecar, minibus) أو اجمع لها صوراً؛ لا يمكن الحكم عليها الآن.
4. صور يمنية مسمّاة للاختبار فقط (≥200): هي القياس الوحيد الذي يعكس بيئتك.

ملاحظة صدق: جُرّبت منطق التقسيم والتجميع والمطابقة محلياً على بيانات اصطناعية، أما التحميل من Hugging Face والتدريب على GPU فلم أستطع تشغيلهما هنا؛ إن ظهر خطأ في تحميل `class_mapping.csv` فأرسله لي وأعدّل المحلّل.